# Question-guided evidence selection + matched verifier (Colab)

This is an **experimental main-method candidate**, not an established improvement. It tests whether asking two claim-specific verification questions helps select *decisive* evidence from the existing BM25 top 100, and whether concise, verbatim excerpts help a classifier read it. It keeps the same official AVeriTeC train/dev split, top-three evidence limit, URL-proxy retrieval metrics (386 judged dev claims), and four-way verdict metrics (500 dev claims).

The selector sees only the claim and the saved BM25 candidates. It never sees gold answers, URLs of gold evidence, verdict labels, or dev qrels. Qrels are opened only after rankings are complete. This method **cannot recover anything absent from BM25 top 100**. It uses Qwen3-4B to make questions, a zero-shot MS MARCO MiniLM cross-encoder to score a 32-passage shortlist, per-question rank fusion, and source-diverse top-three selection. Both the original passages and exact-source sentence excerpts are saved, so two matched DistilBERT verifier runs can isolate the excerpt effect.

**Before Colab:** commit and push this notebook and `scripts/run_question_guided_retrieval.py` plus `src/retrieval/question_guided.py`. Choose a GPU runtime (T4 or better) and do not run another GPU experiment in parallel. All long stages display progress and checkpoint to Drive; rerun the same cell after interruption. The first 20-claim smoke test uses **train only** and is not used to choose hyperparameters. The full run can take hours; the verifier runs are shorter.


In [ ]:
import os, sys, subprocess, gc, json, runpy, time
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-questionguided')
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has local edits; save them before pulling.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-local-llm-colab.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Switch Colab to a GPU runtime before starting'
print('GPU:', torch.cuda.get_device_name(0))
print('Repo revision:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
for name in ('scripts/run_question_guided_retrieval.py', 'scripts/train_transformer_verifier.py', 'scripts/run_local_llm_baseline.py', 'src/retrieval/question_guided.py'):
    assert (REPO_DIR / name).is_file(), f'Missing {name}: commit/push locally, then rerun setup'


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
TRAIN_CLAIMS, DEV_CLAIMS = PROCESSED / 'train.jsonl', PROCESSED / 'dev.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
TOP100_ROOT = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1/bm25'
TRAIN100 = TOP100_ROOT / 'train_bm25_top100_v2/train_rankings.jsonl'
DEV100 = TOP100_ROOT / 'dev_bm25_top100_v2/dev_rankings.jsonl'
ROOT = DATA_ROOT / 'outputs/initial_ml_solution_question_guided_v1'
SMOKE_DIR = ROOT / 'retrieval/train_smoke20'
TRAIN_DIR = ROOT / 'retrieval/train_qwen_ce32_v1'
DEV_DIR = ROOT / 'retrieval/dev_qwen_ce32_v1'
for path in (TRAIN_CLAIMS, DEV_CLAIMS, DEV_QRELS, TRAIN100, DEV100):
    if not path.is_file(): raise FileNotFoundError(path)
def run_script(label, filename, arguments):
    script = REPO_DIR / 'scripts' / filename
    if not script.is_file(): raise FileNotFoundError(script)
    old_argv = sys.argv[:]
    sys.argv = [str(script), *map(str, arguments)]
    print(f'\n=== {label} ===', flush=True)
    started = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = old_argv
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'{label}: {(time.monotonic()-started)/60:.1f} minutes this invocation', flush=True)
COMMON = ['--shortlist-k', '32', '--bm25-keep', '12', '--batch-size', '16', '--max-seq-length', '384', '--excerpt-chars', '500', '--checkpoint-every', '1', '--seed', '718', '--resume']


## 1. Train-only smoke test
This checks Qwen/4-bit loading, cross-encoder scoring, checkpoints, and output shape. The 20 claims are not a validation set and its score is not a performance estimate. After it completes, continue to the full train cell.


In [ ]:
run_script('Question-guided train smoke (20 claims)', 'run_question_guided_retrieval.py', [
    '--input-rankings', TRAIN100, '--claims', TRAIN_CLAIMS, '--split', 'train',
    '--output-dir', ROOT / 'retrieval', '--run-name', 'train_smoke20', '--max-claims', '20', *COMMON,
])
assert (SMOKE_DIR / 'train_full_rankings.jsonl').is_file()
assert (SMOKE_DIR / 'train_excerpt_rankings.jsonl').is_file()


## 2. Full train selection
This is the longer stage. It produces matched train evidence without using train gold evidence. All 3,067 usable train claims should be present before continuing. Restart the same cell after a Colab interruption; it skips checkpointed claims.


In [ ]:
run_script('Question-guided retrieval / train', 'run_question_guided_retrieval.py', [
    '--input-rankings', TRAIN100, '--claims', TRAIN_CLAIMS, '--split', 'train',
    '--output-dir', ROOT / 'retrieval', '--run-name', 'train_qwen_ce32_v1', *COMMON,
])
train_report = json.loads((TRAIN_DIR / 'train_metrics.json').read_text())
assert train_report['finalized'] and train_report['claim_count'] == 3067


## 3. Full dev selection and fixed retrieval evaluation
Dev qrels are passed only for evaluation after rankings are frozen. Do not alter question prompts or shortlist parameters after looking at the dev result; a changed method requires a fresh validation protocol.


In [ ]:
run_script('Question-guided retrieval / dev', 'run_question_guided_retrieval.py', [
    '--input-rankings', DEV100, '--claims', DEV_CLAIMS, '--split', 'dev', '--qrels', DEV_QRELS,
    '--output-dir', ROOT / 'retrieval', '--run-name', 'dev_qwen_ce32_v1', *COMMON,
])
dev_retrieval = json.loads((DEV_DIR / 'dev_metrics.json').read_text())
assert dev_retrieval['finalized'] and dev_retrieval['claim_count'] == 500
print('URL-proxy hit@3:', dev_retrieval['evidence_hit_at_3'], 'MRR@3:', dev_retrieval['mrr_at_3'])
print('BM25 hit@3:', dev_retrieval['bm25_reference_hit_at_3'], 'MRR@3:', dev_retrieval['bm25_reference_mrr_at_3'])
print('Shortlist ceiling:', dev_retrieval['shortlist_hit_at_32'], 'Top-100 ceiling:', dev_retrieval['bm25_candidate_hit_at_100'])
print('Previous cross-encoder + source diversity hit@3=0.2280')


## 4. Matched verifier: original selected passages
We first train the same class-weighted DistilBERT family and train-only 10% selection procedure used in the prior experiments. This isolates what the new selector does *without* excerpting. Trials and final fitting resume from checkpoints.


In [ ]:
run_script('Matched DistilBERT / full passages', 'train_transformer_verifier.py', [
    '--data-root', DATA_ROOT, '--train-rankings', TRAIN_DIR / 'train_full_rankings.jsonl',
    '--eval-rankings', DEV_DIR / 'dev_full_rankings.jsonl',
    '--output-dir', ROOT / 'verifier', '--run-name', 'question_guided_full_v1',
    '--model', 'distilbert-base-uncased', '--learning-rates', '2e-5,5e-5',
    '--epoch-grid', '2,3', '--batch-size', '8', '--gradient-accumulation-steps', '2',
    '--checkpoint-steps', '100', '--seed', '718', '--resume',
])


## 5. Matched verifier: exact-source excerpts
The rankings contain the **same passage IDs** as the full-passage run. Only the text fed to DistilBERT changes. Excerpts are selected from source text with no generated factual answers; this is still an imperfect heuristic, so compare results rather than assuming it helps.


In [ ]:
run_script('Matched DistilBERT / exact-source excerpts', 'train_transformer_verifier.py', [
    '--data-root', DATA_ROOT, '--train-rankings', TRAIN_DIR / 'train_excerpt_rankings.jsonl',
    '--eval-rankings', DEV_DIR / 'dev_excerpt_rankings.jsonl',
    '--output-dir', ROOT / 'verifier', '--run-name', 'question_guided_excerpt_v1',
    '--model', 'distilbert-base-uncased', '--learning-rates', '2e-5,5e-5',
    '--epoch-grid', '2,3', '--batch-size', '8', '--gradient-accumulation-steps', '2',
    '--checkpoint-steps', '100', '--seed', '718', '--resume',
])


## 6. Optional: same Qwen verifier, new evidence
This tests whether the existing zero-shot Qwen verdict prompt can use the new top-three excerpts better than the prior BM25 top-three evidence. It is slower than DistilBERT but resumable after each claim. It does **not** train Qwen. Set the flag to `False` if Colab time is limited; the matched DistilBERT comparisons above are the core experiment.


In [ ]:
RUN_QWEN_VERIFIER = True
if RUN_QWEN_VERIFIER:
    run_script('Qwen verifier / question-guided excerpts', 'run_local_llm_baseline.py', [
        '--claims', DEV_CLAIMS, '--rankings', DEV_DIR / 'dev_excerpt_rankings.jsonl',
        '--qrels', DEV_QRELS, '--evidence-mode', 'retrieved_top3', '--variant', 'classify_top3',
        '--output-dir', ROOT / 'llm', '--run-name', 'question_guided_excerpt_qwen_v1',
        '--model', 'Qwen/Qwen3-4B', '--max-new-tokens', '256', '--seed', '718',
    ])


## 7. Compare with frozen baselines
A higher URL-proxy hit@3 alone is not enough: compare macro-F1, accuracy, both safety errors, and the four per-class F1 scores. The old BM25 + DistilBERT and cross-encoder + diversity + matched DistilBERT numbers below are from prior runs on these same 500 dev claims. Do not claim deployment readiness based on these offline results.


In [ ]:
from IPython.display import display
import pandas as pd
def verdict_row(name, metrics, retrieval=None):
    errors = metrics['decision_error_rates']
    row = {'system': name, 'URL-proxy hit@3': retrieval, 'macro-F1': metrics['macro_f1'],
           'accuracy': metrics['accuracy'],
           'false reassurance': errors['false_reassurance_rate'],
           'false alarm': errors['false_alarm_rate']}
    for label in ('Supported', 'Refuted', 'Not Enough Evidence', 'Conflicting Evidence'):
        row[label + ' F1'] = metrics['per_class'][label]['f1-score']
    return row
full = json.loads((ROOT / 'verifier/question_guided_full_v1/final_dev_metrics.json').read_text())
excerpt = json.loads((ROOT / 'verifier/question_guided_excerpt_v1/final_dev_metrics.json').read_text())
rows = [
    {'system': 'BM25 + Qwen3-4B (prior)', 'URL-proxy hit@3': 0.1968911917,
     'macro-F1': 0.3346, 'accuracy': 0.496, 'false reassurance': 0.220, 'false alarm': 0.058},
    {'system': 'BM25 + DistilBERT (prior)', 'URL-proxy hit@3': 0.1968911917,
     'macro-F1': 0.3469294613, 'accuracy': 0.520, 'false reassurance': 0.148, 'false alarm': 0.156},
    {'system': 'Answer-aligned cross-encoder + diversity + matched DistilBERT (prior)',
     'URL-proxy hit@3': 0.2280, 'macro-F1': 0.3455, 'accuracy': 0.512,
     'false reassurance': 0.162, 'false alarm': 0.164},
    verdict_row('Question-guided + full passages', full, dev_retrieval['evidence_hit_at_3']),
    verdict_row('Question-guided + exact-source excerpts', excerpt, dev_retrieval['evidence_hit_at_3']),
]
qwen_metrics_path = ROOT / 'llm/question_guided_excerpt_qwen_v1/classify_top3_metrics.json'
if qwen_metrics_path.is_file():
    qwen_new = json.loads(qwen_metrics_path.read_text())
    assert qwen_new['claim_count'] == 500, 'Qwen did not finish all 500 dev claims; rerun its cell to retry errors'
    rows.append(verdict_row('Question-guided excerpts + Qwen', qwen_new, dev_retrieval['evidence_hit_at_3']))
display(pd.DataFrame(rows).round(4))
print('Judged retrieval claims:', dev_retrieval['judged_claim_count'])
print('Question fallback count:', dev_retrieval['question_fallback_count'])
print('If macro-F1 or safety errors do not improve, report this as a negative result and keep the strongest prior method.')


## 8. Interpretation and follow-up audit
Read the first 20 rows of the dev checkpoint to inspect the generated questions, selected passages, and excerpts. Do not use this audit to adjust and then re-score on the same dev split without stating that the comparison became exploratory. A true user-facing assistant still needs human review of source quality, passage relevance, and the impact on trust decisions. If this method helps retrieval but not verdicts, the bottleneck is likely evidence interpretation, label definitions, or non-decisive excerpts; if neither improves, spend effort on first-stage coverage rather than more reranking.


In [ ]:
from itertools import islice
from src.utils.io import iter_jsonl
for row in islice(iter_jsonl(DEV_DIR / 'dev_question_guided.partial.jsonl'), 3):
    print('\n', row['claim_id'], row['claim'][:180])
    print('Questions:', row['questions'])
    for item in row['retrieved']:
        print('  ', item['passage_id'], str(item.get('url') or '')[:100], item['excerpt_text'][:240])
